In [ ]:
import pandas as pd

# Apresentando o dataset
df = pd.read_csv('prova3 i.a/car_price_prediction_with_missing.csv')
display(df)

In [ ]:
# Verificando o tipo de dado de cada atributo
df.dtypes

In [ ]:
# Apresentando se contém dados nulos e a quantidade por coluna
df.isnull().sum()

In [ ]:
# Verificando se os nulos são linhas inteiras vazias ou dados dispersos
linhas_completamente_vazias = df.isnull().all(axis=1).sum()
linhas_com_algum_nulo = df.isnull().any(axis=1).sum()

print(f"Linhas com TODOS os valores nulos (completamente vazias): {linhas_completamente_vazias}")
print(f"Linhas com pelo menos UM valor nulo: {linhas_com_algum_nulo}")

if linhas_completamente_vazias == linhas_com_algum_nulo:
    print("\nCONCLUSÃO: Todos os nulos estão concentrados em linhas inteiras que estão totalmente em branco!")
else:
    print("\nCONCLUSÃO: Existem valores nulos dispersos pelo dataset (alguns dados preenchidos e outros faltando na mesma linha).")

In [ ]:
# Removendo as linhas com valores nulos do dataset
df_limpo = df.dropna()

print(f"Número de linhas ANTES da limpeza: {df.shape[0]}")
print(f"Número de linhas DEPOIS da limpeza: {df_limpo.shape[0]}")
print(f"Total de linhas nulas apagadas: {df.shape[0] - df_limpo.shape[0]}\n")

# Verificação final para garantir que não sobrou nenhum nulo
print("Quantidade de nulos após a limpeza:")
print(df_limpo.isnull().sum())

In [ ]:
# Analisando se os dados numéricos estão normalizados
print("Análise das escalas numéricas ANTES da Normalização:")
colunas_numericas = df_limpo.select_dtypes(include=['int64', 'float64']).columns
colunas_para_normalizar = [col for col in colunas_numericas if col not in ['Car ID', 'Price']]

display(df_limpo[colunas_para_normalizar].describe().loc[['min', 'max']])

print("\nCONCLUSÃO: Os dados numéricos NÃO estão normalizados.")

In [ ]:
# Normalizando as colunas numéricas MANUALMENTE (sem o uso do scikit-learn)
df_normalizado = df_limpo.copy()

for col in colunas_para_normalizar:
    valor_minimo = df_normalizado[col].min()
    valor_maximo = df_normalizado[col].max()
    df_normalizado[col] = (df_normalizado[col] - valor_minimo) / (valor_maximo - valor_minimo)

print("✅ Dados numéricos normalizados com sucesso (O Preço NÃO foi normalizado)!\n")
colunas_verificacao = colunas_para_normalizar + ['Price']
display(df_normalizado[colunas_verificacao].describe().loc[['min', 'max']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# MATRIZ DE CORRELAÇÃO DE TODAS AS VARIÁVEIS (Gráfico Quadrado)
sns.set_theme(style="white")

df_matriz = df_limpo.drop(columns=['Car ID']).copy()
for col in df_matriz.select_dtypes(include=['object']).columns:
    df_matriz[col] = df_matriz[col].astype('category').cat.codes

matriz_correlacao = df_matriz.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_correlacao, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, 
            linewidths=0.5, linecolor='white')
plt.title('Matriz de Correlação das Variáveis', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# GRÁFICO EXTRA: PAIRPLOT (Matriz de Dispersão Multivariada)
# Este gráfico junta várias variáveis numéricas contra o Preço ao mesmo tempo,
# e ainda usa as CORES para mostrar a influência de uma variável categórica (Condição do carro).

import warnings
warnings.filterwarnings('ignore') # Ocultar alertas visuais do Seaborn

plt.figure(figsize=(12, 6))

# Vamos plotar Ano, Tamanho do Motor e Quilometragem contra o Preço.
# A cor dos pontos (hue) vai representar a Condição (New, Like New, Used) e o marcador (style) vai representar o Combustível.
g = sns.pairplot(df_limpo, 
                 x_vars=['Year', 'Engine Size', 'Mileage'], 
                 y_vars=['Price'], 
                 hue='Condition', 
                 palette='Set1', 
                 height=5, 
                 aspect=1.2, 
                 kind='scatter',
                 plot_kws={'alpha':0.6, 's':80, 'edgecolor':'k'})

g.fig.suptitle('Matriz de Dispersão: Variáveis Numéricas vs Preço (Colorido pela Condição)', y=1.08, fontsize=16)
plt.show()